# Melting Point of chemical molecules using Graph Neural Network

##### Made use of BradleyMeltingPointDataset for the model.

## Downloading necessary libraries

In [ ]:
!pip install "numpy<2.0"


In [ ]:
!pip install rdkit


In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
# Install PyTorch Geometric dependencies and core
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.2.0+cpu.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.2.0+cpu.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.2.0+cpu.html
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-2.2.0+cpu.html
!pip install torch-geometric

In [ ]:
!nvidia-smi

In [ ]:
## Importing necessary libraries
import pandas as pd
import rdkit
import scipy
from rdkit import Chem
from rdkit.Chem import Draw
import torch
import matplotlib.pyplot as plt
import torch_geometric

## Uploading the dataset from google drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


In [ ]:
df = pd.read_csv("/content/gdrive/MyDrive/MeltingPointDataset.csv")
df = df[['name','smiles','mpC']]
df

In [ ]:
df

In [ ]:
df['name'] = df['name'].str.lower()
df['name'].duplicated().sum()/len(df)

In [ ]:
df['smiles'].duplicated().sum()/len(df)

In [ ]:
df_new= df.drop_duplicates(subset=['name','smiles'], keep='first', inplace=False, ignore_index=False)

In [ ]:
df_new.shape

In [ ]:
df_new

In [ ]:
df_new = df_new.reset_index(drop=True)
df_new

## Cleaning up the datasets

#### We are kekulising the smiles so as to use it

In [ ]:
def kekule_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
        Chem.SanitizeMol(mol)  # Run checks
        Chem.Kekulize(mol, clearAromaticFlags=True)
        return Chem.MolToSmiles(mol, kekuleSmiles=True)
    except Exception as e:
        return None  # or log: f"{smiles} --> {e}"


In [ ]:
df_new["kekule"] = df_new["smiles"].apply(kekule_smiles)
df_new['kekule']

In [ ]:
null_count = df_new['kekule'].isna().sum()
print(f"Number of null values in the kekule column: {null_count}")

##### Dropping the null values from the kekulised smiles column

In [ ]:
df_new = df_new.dropna(subset=['kekule'])
df_new = df_new.reset_index(drop=True)
df_new

In [ ]:
df_new["kekule"][0]

##### converting the smiles to mol file

In [ ]:
df_new["Mol"] = df_new["kekule"].apply(Chem.MolFromSmiles)
df_new['Mol']

In [ ]:
df_new["Mol"][0]

In [ ]:
df_new["Mol"][1000]

In [ ]:
df_new["Mol"][10000]

In [ ]:
df_new["Mol"][2000]

##### Indexing the diffrent atoms present in a molecule

In [ ]:
def get_idx(molecule):
    for atom in molecule.GetAtoms():
        #print (atom)
        idx_atom = atom.GetIdx()
        #print(idx_atom)
        atom.SetAtomMapNum(idx_atom)
    return (molecule)

df_new["idx_mol"] = df_new["Mol"].apply(lambda x: get_idx(x))
df_new["idx_mol"][15892]

##### Finding out the edges of the molecule

In [ ]:
def find_edge(molecule):
    edges = []
    for bond in molecule.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges.append([i,j])
        edges.append([j,i])
    return edges

df_new['edge'] = df_new['idx_mol'].apply(lambda x:find_edge(x))
df_new["edge"][15892]

##### Gettign the adjacency matrix

In [ ]:
from rdkit.Chem import rdmolops

In [ ]:
def adj_matrix(molecule):
    x = rdkit.Chem.rdmolops.GetAdjacencyMatrix(molecule)

    return torch.tensor(x, dtype = torch.long)# check with google tensor?

df_new["adj_matrix"] = df_new["idx_mol"].apply(lambda x: adj_matrix(x))
df_new["adj_matrix"][2000]

In [ ]:
df_new["edge"][1]

In [ ]:
# x = list(zip([1,2],[2,1],[2,3],[3,4]))
# x

In [ ]:
# row, col = list(zip(*df_new["edge"][2000]))
# #print(row)
# #print(col)
# x = torch.tensor([row, col])
# print(x)


In [ ]:
empty_edges_mask = df_new["edge"].apply(lambda x: len(x) == 0)
empty_edges_mask
empty_rows = df_new[empty_edges_mask]
#empty_rows.head()

print(f"Number of graphs with empty edges: {len(empty_rows)}")
print(empty_rows)


##### Converting the edge index to COOformat

In [ ]:
def coo_edge_index(edges):
    if edges:
        row, col = list(zip(*edges))
        indices = torch.tensor([row, col], dtype=torch.long)
        return indices
    else:
        return torch.empty((2, 0), dtype=torch.long)  # Return empty edge_index if no edges

df_new["coo_matrix"] = df_new["edge"].apply(coo_edge_index)

In [ ]:
df_new.head()

In [ ]:
df_new["Mol"][1]

##### Getting the node feature matrix: feature 1: atomic number; feature 2: degree of the atoms; feature 3: Number of implicit hydrogen; feature 4: checking whether the atom is part of ring and feature 5: whether atom is aromatic or not

In [ ]:
def node_feature(atom):
    return [atom.GetAtomicNum(),
          atom.GetDegree(),
          atom.GetNumImplicitHs(),
            atom.IsInRing(),
            atom.GetIsAromatic()]

def node_feature_matrix(molecule):
    node_matrix = []
    for atoms in molecule.GetAtoms():
        node_matrix.append(node_feature(atoms))
    return torch.tensor(node_matrix, dtype = torch.float)

df_new["node_matrix"] = df_new["idx_mol"].apply(lambda x: node_feature_matrix(x))
#df.head()
df_new["node_matrix"][1]

In [ ]:
# first_col = df_new["node_matrix"][1][:,0]
# first_col

In [ ]:
df_new["node_matrix"]

In [ ]:
#for matrix in df_new["node_matrix"]:
max_val = max(matrix[:, 0].max().item() for matrix in df_new['node_matrix'])
print("The highest atomic number present in the dataset is:", max_val)



In [ ]:
for i in range(1,83):
  count = sum((t[:, 0] == i).any().item() for t in df_new["node_matrix"])

  print(f"Number of compounds with element of atomic number {i} present is: {count}")


In [ ]:
df_new["node_matrix"][0]

In [ ]:
len(df_new)

In [ ]:
import torch_geometric
from torch_geometric.data import Data

## Creating dataset for the model

In [ ]:
dataset = []
for _, row in df_new.iterrows():
    x = row["node_matrix"]
    edge_index = row["coo_matrix"]
    y = torch.tensor([row["mpC"]], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, y=y)
    dataset.append(data)

dataset


## Splitting into train and test dataset

In [ ]:
import random

random.seed(13)
data = random.sample(dataset, len(dataset))

train_dataset = dataset[:25000]
test_dataset = dataset[25000:]

print(f'Number of training graphs: {len(train_dataset)}')
print(f'Number of test graphs: {len(test_dataset)}')

## Splitting to batches using dataloader

In [ ]:
from torch_geometric.loader import DataLoader
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataloaders: {train_loader, test_loader}")
print(f"Length of train loader: {len(train_loader)} batches of {BATCH_SIZE}")
print(f"Length of test loader: {len(test_loader)} batches of {BATCH_SIZE}")


# for step, data in enumerate(train_loader):
#     print(f'Step {step + 1}:')
#     print('=======')
#     print(f'Number of graphs in the current training batch: {data.num_graphs}')
#     print(data)
#     print()

# for step, data in enumerate(test_loader):
#     print(f'Step {step + 1}:')
#     print('=======')
#     print(f'Number of graphs in the current testing batch: {data.num_graphs}')
#     print(data)
#     print()

In [ ]:
print(len(train_loader))

In [ ]:
for i, batch in enumerate(train_loader):
    print(f"\n🔹 Batch {i+1}")
    print(f"Batch {batch}")
    print("Number of nodes:", batch.x.size(0))
    print("Number of edges:", batch.edge_index.size(1))
    print("Target shape:", batch.y.shape)
    print("Graphs in batch:", batch.num_graphs)
    print("Batch vector shape:", batch.batch.shape)

    if i == 4:   # Stop after 5 batches
        break


In [ ]:
for i, batch in enumerate(test_loader):
    print(f"\n🔹 Batch {i+1}")
    print(f"Batch {batch}")
    print("Number of nodes:", batch.x.size(0))
    print("Number of edges:", batch.edge_index.size(1))
    print("Target shape:", batch.y.shape)
    print("Graphs in batch:", batch.num_graphs)
    print("Batch vector shape:", batch.batch.shape)

    if i == 4:   # Stop after 5 batches
        break

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device


In [ ]:
from torch import nn
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool as gmp, global_max_pool as gap

## Model_0:(Took the same model used for ESOL dataset)

In [ ]:
class GCNModelV0(torch.nn.Module):
  def __init__(self, input_unit:int, hidden_unit:int, output_unit:int):
    super().__init__()
    self.conv1 = GCNConv(input_unit, hidden_unit)
    self.conv2 = GCNConv(hidden_unit, hidden_unit)
    self.conv3 = GCNConv(hidden_unit, hidden_unit)
    self.conv4 = GCNConv(hidden_unit, hidden_unit)
    self.linear = Linear(hidden_unit*2, 1)


  def forward(self, x, edge_index, batch):
    x = self.conv1(x, edge_index)
    x = F.tanh(x)
    x = self.conv2(x, edge_index)
    x = F.tanh(x)
    x = self.conv3(x, edge_index)
    x = F.tanh(x)
    x = self.conv4(x, edge_index)
    x = F.tanh(x)

    x = torch.cat([gmp(x, batch),
                            gap(x, batch)], dim=1)



    #x = F.dropout(x, p=0.5, training=self.training)

    x = self.linear(x)
    return x




torch.manual_seed(42)
model_0 = GCNModelV0(input_unit= 5, hidden_unit= 64, output_unit= 1)
print (model_0)

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(params = model_0.parameters(), lr = 0.0007)

In [ ]:
from timeit import default_timer as timer
def print_train_time(start: float, end: float):
    train_time =  end - start
    print(f"Total time taken for training: {train_time:.6f}seconds")
    return train_time

In [ ]:
def train_loop(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):
    train_loss = 0

    for batch in train_loader:
        model.train()

        ##Forwars pass
        y_pred = model(batch.x.float(), batch.edge_index, batch.batch)

        ## Calculate the loss
        loss = loss_fn(y_pred, batch.y.float())
        train_loss += loss


        ## Optimizer zero grad
        optimizer.zero_grad()

        ## Loss backward
        loss.backward()

        ## optimer step
        optimizer.step()

    train_loss /= len(train_loader)

    print(f" Train loss: {train_loss}")

In [ ]:
def test_loop(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              optimizer: torch.optim.Optimizer):
    test_loss = 0

    for batch in test_loader:

        test_pred = model(batch.x.float(), batch.edge_index, batch.batch)

        ## Calculate loss
        loss = loss_fn(test_pred, batch.y.float())
        test_loss += loss


    test_loss /= len(test_loader)

    print(f"Test_loss: {test_loss}")


In [ ]:
from tqdm.auto import tqdm ## FOR KEEPING THE progress bar
torch.manual_seed(42)
start_time = timer()
epochs = 20000

for epoch in tqdm(range(epochs)):
  if epoch % 1000 == 0:
    print(f"Epoch: {epoch}------------")
    train_loop(model_0, train_loader, loss_fn, optimizer)


    model_0.eval()
    with torch.inference_mode():
        test_loop(model_0, test_loader, loss_fn, optimizer)

end_time = timer()
total_time = end_time - start_time
print(f"The total time taken for the model to run is: {total_time}")



##My God!!!! Such a poor model.... Definitely needto improve it